# 01 — Train the YOLO11 object-segmentation model

This notebook prepares the Roboflow YOLO-format export and trains the first
RIDAC stage: an instance-segmentation model that identifies each manufactured
object and predicts its product category.

## Pipeline position

`VisA images → Roboflow polygon annotations → this notebook → trained YOLO11 checkpoint`

## Inputs

- `VisA_Object_Segmentation.v3i.yolov11/data.yaml`
- Train, validation, and test images exported by Roboflow
- Matching YOLO segmentation label files
- `yolo11n-seg.pt`, used as the pretrained starting checkpoint

## Main output

Ultralytics creates a run directory containing training curves, validation
plots, and `weights/best.pt`. That best checkpoint is used by notebook 02.

> **Important:** run the cells in the displayed order. The annotation audit
> must happen before training. The cleanup cell deletes only the explicitly
> listed malformed image/label pairs, so review the list before running it.


## 1. Audit the exported segmentation labels

A valid YOLO segmentation row contains a class ID followed by at least three
polygon points. A row containing exactly five values is detection-box format,
not polygon format, and would make the segmentation dataset inconsistent.

This audit is read-only: it prints suspicious files but does not change them.
The expected result after cleanup is `Remaining bad files: 0` for every split.


In [16]:
# Audit every YOLO label file for box-format rows
from pathlib import Path

dataset = Path("VisA_Object_Segmentation.v3i.yolov11")

for split in ["train", "valid", "test"]:

    # Roboflow uses the folder name `valid` for validation data.
    label_folder = dataset / split / "labels"

    print("\nChecking:", split)

    bad = 0

    for txt in label_folder.glob("*.txt"):

        for line in txt.read_text().splitlines():

            values = line.split()

            # Five values indicate class + bounding box, not a polygon.
            if len(values) == 5:
                print("Bad:", txt.name)
                bad += 1

    print("Remaining bad files:", bad)



Checking: train
Bad: capsules_Anomaly_050_JPG.rf.42085394fda7f881031e5d5e59148bac.txt
Bad: 044_JPG.rf.fefe4306c6128a5df1d74513decba2a0.txt
Bad: capsules_Normal_357_JPG.rf.5ee60e0deb8be6f0b58ca6f7b5251637.txt
Bad: pipe_fryum_Normal_472_JPG.rf.d1ebcf9c2e63bf840a328b76b380e218.txt
Bad: 009_JPG.rf.c2bb54c88a56a039197839935240d523.txt
Bad: 000_JPG.rf.e976a93cc382400490e7ac93f7ed3d6d.txt
Bad: capsules_Anomaly_054_JPG.rf.22408f4e739c27764a4e9a87b3643aa7.txt
Bad: 040_JPG.rf.0421b86c4266ec9c15b86d3ee0ae84b0.txt
Remaining bad files: 8

Checking: valid
Bad: pipe_fryum_Normal_333_JPG.rf.31e84e651366e992a4de3ab2bdeff256.txt
Bad: 045_JPG.rf.8a445451660f7cc5e949eca9eb131f90.txt
Bad: 010_JPG.rf.3200c19cd7a42ebbf953d69cdf198663.txt
Bad: 000_JPG.rf.54cbd225f8bc3944d0758cbcf6bb7f40.txt
Remaining bad files: 4

Checking: test
Remaining bad files: 0


## 2. Remove the known malformed image/label pairs

This project identified a small fixed set of records that contained box labels
inside the segmentation export. The cell removes both the label and its image
counterpart so each split remains aligned.

Safety notes:

- The operation is destructive inside the local exported dataset.
- It is idempotent: missing files are skipped on later runs.
- Re-run the audit above afterward and confirm that no five-value rows remain.
- If your Roboflow export differs, update the list instead of deleting unrelated
  files.


In [17]:
# Delete only the reviewed malformed image/label pairs
from pathlib import Path

dataset = Path("./VisA_Object_Segmentation.v3i.yolov11")

# These stems were manually confirmed as invalid in this export.
bad_files = [
    # Train
    "capsules_Anomaly_050_JPG.rf.42085394fda7f881031e5d5e59148bac",
    "044_JPG.rf.fefe4306c6128a5df1d74513decba2a0",
    "capsules_Normal_357_JPG.rf.5ee60e0deb8be6f0b58ca6f7b5251637",
    "pipe_fryum_Normal_472_JPG.rf.d1ebcf9c2e63bf840a328b76b380e218",
    "009_JPG.rf.c2bb54c88a56a039197839935240d523",
    "000_JPG.rf.e976a93cc382400490e7ac93f7ed3d6d",
    "capsules_Anomaly_054_JPG.rf.22408e0deb8be6f0b58ca6f7b5251637",
    "040_JPG.rf.0421b86c4266ec9c15b86d3ee0ae84b0",

    # Validation
    "pipe_fryum_Normal_333_JPG.rf.31e84e651366e992a4de3ab2bdeff256",
    "045_JPG.rf.8a445451660f7cc5e949eca9eb131f90",
    "010_JPG.rf.3200c19cd7a42ebbf953d69cdf198663",
    "000_JPG.rf.54cbd225f8bc3944d0758cbcf6bb7f40",
]


for split in ["train", "valid", "test"]:

    image_dir = dataset / split / "images"
    label_dir = dataset / split / "labels"

    if not image_dir.exists():
        continue

    print("\nProcessing:", split)

    for name in bad_files:

        # Remove the annotation first, then its matching image.
        label_file = label_dir / f"{name}.txt"

        if label_file.exists():
            print("Deleting label:", label_file.name)
            label_file.unlink()

        # Remove image
        for ext in [".jpg", ".jpeg", ".png"]:

            image_file = image_dir / f"{name}{ext}"

            if image_file.exists():
                print("Deleting image:", image_file.name)
                image_file.unlink()


print("\nFinished removing bad files.")



Processing: train
Deleting label: capsules_Anomaly_050_JPG.rf.42085394fda7f881031e5d5e59148bac.txt
Deleting image: capsules_Anomaly_050_JPG.rf.42085394fda7f881031e5d5e59148bac.jpg
Deleting label: 044_JPG.rf.fefe4306c6128a5df1d74513decba2a0.txt
Deleting image: 044_JPG.rf.fefe4306c6128a5df1d74513decba2a0.jpg
Deleting label: capsules_Normal_357_JPG.rf.5ee60e0deb8be6f0b58ca6f7b5251637.txt
Deleting image: capsules_Normal_357_JPG.rf.5ee60e0deb8be6f0b58ca6f7b5251637.jpg
Deleting label: pipe_fryum_Normal_472_JPG.rf.d1ebcf9c2e63bf840a328b76b380e218.txt
Deleting image: pipe_fryum_Normal_472_JPG.rf.d1ebcf9c2e63bf840a328b76b380e218.jpg
Deleting label: 009_JPG.rf.c2bb54c88a56a039197839935240d523.txt
Deleting image: 009_JPG.rf.c2bb54c88a56a039197839935240d523.jpg
Deleting label: 000_JPG.rf.e976a93cc382400490e7ac93f7ed3d6d.txt
Deleting image: 000_JPG.rf.e976a93cc382400490e7ac93f7ed3d6d.jpg
Deleting label: 040_JPG.rf.0421b86c4266ec9c15b86d3ee0ae84b0.txt
Deleting image: 040_JPG.rf.0421b86c4266ec9c15b8

## 3. Train YOLO11 segmentation

The model starts from the lightweight `yolo11n-seg.pt` checkpoint and fine-tunes
for up to 100 epochs at 640 × 640 resolution.

### Key parameters

| Parameter | Value | Meaning |
|---|---:|---|
| `epochs` | 100 | Maximum training epochs |
| `imgsz` | 640 | Input resolution used by the Roboflow export |
| `batch` | 16 | Images per optimizer step |
| `optimizer` | AdamW | Optimizer used for fine-tuning |
| `lr0` | 0.001 | Initial learning rate |
| `patience` | 30 | Early-stopping patience |
| `amp` | True | Mixed-precision CUDA training |

If no CUDA GPU is available, change `device=0` to `device="cpu"` and reduce the
batch size. CPU training is supported but will be substantially slower.


In [19]:
# Load the pretrained YOLO11 segmentation model and fine-tune it
from ultralytics import YOLO
import torch

# Report the compute device before starting a potentially long run.
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# Initialize from Ultralytics' nano segmentation checkpoint.
model = YOLO("yolo11n-seg.pt")


# Ultralytics writes checkpoints and plots under runs/segment/.
model.train(
    data="./VisA_Object_Segmentation.v3i.yolov11/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,          # GPU ID
    workers=8,         # data loading threads
    optimizer="AdamW",
    lr0=0.001,
    patience=30,
    amp=True           # mixed precision training (faster GPU training)
)


CUDA available: True
GPU: NVIDIA RTX 6000 Ada Generation
Ultralytics 8.4.95 🚀 Python-3.11.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48637MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./VisA_Object_Segmentation.v3i.yolov11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, mu

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x705d307c9a90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.03

## 4. Verify the training output

After training:

1. Open the generated `runs/segment/train*/` directory.
2. Review `results.png`, precision/recall curves, and mask examples.
3. Confirm that `weights/best.pt` exists.
4. Use that checkpoint in notebook
   `02_evaluate_segment_and_extract_crops.ipynb`.

The production repository already contains the selected final checkpoint at
`../models/yolo11_seg_best.pt`; rerunning this notebook is optional.
